# Publications markdown generator for academicpages

Takes a set of bibtex of publications and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)).

The core python code is also in `pubsFromBibs.py`.
Run either from the `markdown_generator` folder after replacing updating the publist dictionary with:
* bib file names
* specific venue keys based on your bib file preferences
* any specific pre-text for specific files
* Collection Name (future feature)

TODO: Make this work with other databases of citations,
TODO: Merge this with the existing TSV parsing solution

In [26]:
#!pip install pybtex

from pybtex.database.input import bibtex
import pybtex.database.input.bibtex
from time import strptime
import string
import html
import os
import re




In [27]:
#todo: incorporate different collection types rather than a catch all publications, requires other changes to template
publist = {
    "journal":{
        "file": "output.bib",
        "venuekey" : "journal",
        "venue-pretext" : "",
        "collection" : {"name":"publications",
                        "permalink":"/publication/"}
    }
}

In [28]:
import re

def clean_bib(path):
    with open(path, encoding='utf-8') as f:
        text = f.read()

    entries = [e for e in re.split(r'(?m)^\s*(?=@)', text) if e.strip()]
    seen_keys = {}
    seen_dois = set()
    unique = []

    for entry in entries:
        doi_match = re.search(r'doi\s*=\s*[{"]([^}"]+)', entry, re.I)
        if doi_match:
            doi = doi_match.group(1).strip().lower()
            if doi in seen_dois:
                print(path, '- dropped duplicate DOI:', doi)
                continue
            seen_dois.add(doi)

        key_match = re.match(r'\s*@\w+\s*\{\s*([^,\s]+)\s*,', entry)
        if key_match:
            key = key_match.group(1)
            n = seen_keys.get(key.lower(), 0)
            seen_keys[key.lower()] = n + 1
            if n:
                new_key = key + chr(ord('a') + n)
                entry = entry.replace(key + ',', new_key + ',', 1)
                print(path, '- renamed', key, '->', new_key)
        unique.append(entry)

    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(unique))
    print(path, '-', len(unique), 'entries kept of', len(entries))

for pubsource in publist:
    clean_bib(publist[pubsource]['file'])

output.bib - renamed Hartharn_Evans_2024 -> Hartharn_Evans_2024b
output.bib - renamed Hartharn_Evans_2022 -> Hartharn_Evans_2022b
output.bib - 8 entries kept of 8


In [29]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

In [32]:
from pybtex.database.input.bibtex import month_names

macros = dict(month_names)
for m in ['january','february','march','april','may','june','july',
          'august','september','october','november','december','sept']:
    macros[m] = m.capitalize() if m != 'sept' else 'September'

for pubsource in publist:
    parser = bibtex.Parser(macros=macros)
    bibdata = parser.parse_file(publist[pubsource]["file"])

    #loop through the individual references in a given bibtex file
    for bib_id in bibdata.entries:
        #reset default date
        pub_year = "1900"
        pub_month = "01"
        pub_day = "01"

        b = bibdata.entries[bib_id].fields

        try:
            pub_year = f'{b["year"]}'

            #todo: this hack for month and day needs some cleanup
            if "month" in b.keys():
                if(len(b["month"])<3):
                    pub_month = "0"+b["month"]
                    pub_month = pub_month[-2:]
                elif(b["month"] not in range(12)):
                    tmnth = strptime(b["month"][:3],'%b').tm_mon
                    pub_month = "{:02d}".format(tmnth)
                else:
                    pub_month = str(b["month"])
            if "day" in b.keys():
                pub_day = str(b["day"])

            pub_date = pub_year+"-"+pub_month+"-"+pub_day

            #strip out {} as needed (some bibtex entries that maintain formatting)
            clean_title = b["title"].replace("{", "").replace("}","").replace("\\","").replace(" ","-")

            url_slug = re.sub("\\[.*\\]|[^a-zA-Z0-9_-]", "", clean_title)
            url_slug = url_slug.replace("--","-")

            md_filename = (str(pub_date) + "-" + url_slug + ".md").replace("--","-")
            html_filename = (str(pub_date) + "-" + url_slug).replace("--","-")

            #Build Citation from text
            citation = ""

            #citation authors - todo - add highlighting for primary author?
            for author in bibdata.entries[bib_id].persons["author"]:
                citation = citation+" "+author.first_names[0]+" "+author.last_names[0]+", "

            #citation title
            citation = citation + "\"" + html_escape(b["title"].replace("{", "").replace("}","").replace("\\","")) + ".\""

            #add venue logic depending on citation type
            venue = publist[pubsource]["venue-pretext"]+b[publist[pubsource]["venuekey"]].replace("{", "").replace("}","").replace("\\","")

            citation = citation + " " + html_escape(venue)
            citation = citation + ", " + pub_year + "."

            ## YAML variables
            md = "---\ntitle: \""   + html_escape(b["title"].replace("{", "").replace("}","").replace("\\","")) + '"\n'

            md += """collection: """ +  publist[pubsource]["collection"]["name"]

            md += """\npermalink: """ + publist[pubsource]["collection"]["permalink"]  + html_filename

            note = False
            if "note" in b.keys():
                if len(str(b["note"])) > 5:
                    md += "\nexcerpt: '" + html_escape(b["note"]) + "'"
                    note = True

            md += "\ndate: " + str(pub_date)

            md += "\nvenue: '" + html_escape(venue) + "'"

            url = False
            if "url" in b.keys():
                if len(str(b["url"])) > 5:
                    md += "\npaperurl: '" + b["url"] + "'"
                    url = True

            md += "\ncitation: '" + html_escape(citation) + "'"

            md += "\n---"

            ## Markdown description for individual page
            if note:
                md += "\n" + html_escape(b["note"]) + "\n"

            if url:
                md += "\n[Access paper here](" + b["url"] + "){:target=\"_blank\"}\n"
            else:
                md += "\nUse [Google Scholar](https://scholar.google.com/scholar?q="+html.escape(clean_title.replace("-","+"))+"){:target=\"_blank\"} for full citation"

            md_filename = os.path.basename(md_filename)

            with open("./_publications/" + md_filename, 'w', encoding="utf-8") as f:
                f.write(md)
            print(f'SUCESSFULLY PARSED {bib_id}: \"', b["title"][:60],"..."*(len(b['title'])>60),"\"")

        # field may not exist for a reference
        except KeyError as e:
            print(f'WARNING Missing Expected Field {e} from entry {bib_id}: \"', b["title"][:30],"..."*(len(b['title'])>30),"\"")
            continue

SUCESSFULLY PARSED Carr_2026: " Shoaling of internal solitary waves over linear slopes with  ... "
SUCESSFULLY PARSED Hartharn_Evans_2026b: " From concept to action: Driving impact through responsible r ... "
SUCESSFULLY PARSED Hartharn_Evans_2026: " Dynamics of shear instabilities near the ice-shelf–ocean int ... "
SUCESSFULLY PARSED Stastna_2025: " The three-dimensional state of stratified shear instability  ... "
SUCESSFULLY PARSED Hartharn_Evans_2024: " A new approach to understanding fluid mixing in process-stud ... "
SUCESSFULLY PARSED Hartharn_Evans_2024b: " Interactions Between Internal Solitary Waves and Sea Ice  "
SUCESSFULLY PARSED Hartharn_Evans_2022: " Dense pulses formed from fissioning internal waves  "
SUCESSFULLY PARSED Hartharn_Evans_2022b: " Stratification effects on shoaling internal solitary waves  "
